In [4]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import yfinance as yf
import ta
import warnings
warnings.filterwarnings('ignore')

# Configuration graphique professionnelle
plt.rcParams.update({
    'figure.facecolor' : '#0d1117',
    'axes.facecolor'   : '#0d1117',
    'axes.edgecolor'   : '#30363d',
    'axes.labelcolor'  : '#8b949e',
    'text.color'       : '#e6edf3',
    'xtick.color'      : '#8b949e',
    'ytick.color'      : '#8b949e',
    'grid.color'       : '#21262d',
    'grid.linewidth'   : 0.5,
    'figure.dpi'       : 120,
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print("Configuration graphique dark mode activée")

Device : cuda
Configuration graphique dark mode activée


In [5]:
SYMBOL = "SPY"

print(f"Téléchargement {SYMBOL}...")

# Daily — 5 ans pour le HTF
df_1d = yf.download(SYMBOL, start="2020-01-01",
                    interval='1d', progress=False)
df_1d.columns = [col[0] for col in df_1d.columns]

# 1H — maximum 729 jours (limite Yahoo Finance)
df_1h = yf.download(SYMBOL, period="729d",
                    interval='1h', progress=False)
df_1h.columns = [col[0] for col in df_1h.columns]

# Vérification
print(f"Daily : {len(df_1d)} bougies — "
      f"{df_1d.index[0].date()} → {df_1d.index[-1].date()}")

if len(df_1h) > 0:
    print(f"1H    : {len(df_1h)} bougies — "
          f"{df_1h.index[0].date()} → {df_1h.index[-1].date()}")
else:
    print("1H    : échec téléchargement — utilisation 4H")
    df_1h = yf.download(SYMBOL, period="729d",
                        interval='4h', progress=False)
    df_1h.columns = [col[0] for col in df_1h.columns]
    print(f"4H    : {len(df_1h)} bougies chargées")

Téléchargement SPY...
Daily : 1588 bougies — 2020-01-02 → 2026-04-28
1H    : 5071 bougies — 2023-06-01 → 2026-04-28


In [6]:
def add_rsi_ribbon(df):
    """RSI court(7), moyen(14), long(21) + score confluence"""

    df['RSI_7']  = ta.momentum.RSIIndicator(
                        df['Close'], window=7).rsi()
    df['RSI_14'] = ta.momentum.RSIIndicator(
                        df['Close'], window=14).rsi()
    df['RSI_21'] = ta.momentum.RSIIndicator(
                        df['Close'], window=21).rsi()

    # Score confluence -3 à +3
    def score(row):
        s  = 1 if row['RSI_7']  > row['RSI_14'] else -1
        s += 1 if row['RSI_14'] > row['RSI_21'] else -1
        s += 1 if row['RSI_7']  > row['RSI_21'] else -1
        return s

    df['RSI_score'] = df.apply(score, axis=1)

    # Signal textuel
    mapping = { 3:'BULL_MAX', 2:'BULL',  1:'BULL_WEAK',
               -1:'BEAR_WEAK',-2:'BEAR',-3:'BEAR_MAX', 0:'NEUTRAL'}
    df['RSI_signal'] = df['RSI_score'].map(mapping).fillna('NEUTRAL')

    return df

df_1d = add_rsi_ribbon(df_1d)
df_1h = add_rsi_ribbon(df_1h)

# Aperçu
print("RSI Ribbon calculé :")
print(df_1d[['Close','RSI_7','RSI_14','RSI_21',
             'RSI_score','RSI_signal']].tail(5).to_string())
print(f"\nDistribution des signaux (Daily) :")
print(df_1d['RSI_signal'].value_counts().to_string())

RSI Ribbon calculé :
                 Close      RSI_7     RSI_14     RSI_21  RSI_score RSI_signal
Date                                                                         
2026-04-22  711.210022  78.969029  70.728047  65.211941          3   BULL_MAX
2026-04-23  708.450012  72.765086  68.081098  63.562067          3   BULL_MAX
2026-04-24  713.940002  76.964726  70.450050  65.390860          3   BULL_MAX
2026-04-27  715.169983  77.857193  70.969888  65.794710          3   BULL_MAX
2026-04-28  711.414978  68.416217  67.089602  63.422389          3   BULL_MAX

Distribution des signaux (Daily) :
RSI_signal
BULL_MAX     799
BEAR_MAX     568
BULL_WEAK    113
BEAR_WEAK    108


In [7]:
def add_smc_features(df, swing_period=20):
    """
    SSL  : Sell Side Liquidity — swing low récent
    BSL  : Buy Side Liquidity  — swing high récent
    OB   : Order Block         — bougie impulsive
    FVG  : Fair Value Gap      — imbalance de prix
    CHoCH: Change of Character — rupture de structure
    """
    df = df.copy()

    # === SSL / BSL ===
    df['swing_low']    = df['Low'].rolling(swing_period).min()
    df['swing_high']   = df['High'].rolling(swing_period).max()
    df['ssl_distance'] = (df['Close'] - df['swing_low']) / df['Close']
    df['bsl_distance'] = (df['swing_high'] - df['Close']) / df['Close']

    # Proximité zone liquidité (< 0.5% = très proche)
    df['near_ssl'] = (df['ssl_distance'] < 0.005).astype(int)
    df['near_bsl'] = (df['bsl_distance'] < 0.005).astype(int)

    # === ORDER BLOCKS ===
    # Grande bougie + volume fort = potentiel OB
    vol_mean    = df['Volume'].rolling(20).mean()
    ret_std     = df['Close'].pct_change().rolling(20).std()
    body_size   = abs(df['Close'] - df['Open']) / df['Open']

    df['ob_bullish'] = (
        (df['Close'] > df['Open']) &          # bougie verte
        (body_size > 2 * ret_std) &            # grande bougie
        (df['Volume'] > 1.5 * vol_mean)        # volume fort
    ).astype(int)

    df['ob_bearish'] = (
        (df['Close'] < df['Open']) &           # bougie rouge
        (body_size > 2 * ret_std) &            # grande bougie
        (df['Volume'] > 1.5 * vol_mean)        # volume fort
    ).astype(int)

    # === FAIR VALUE GAP ===
    # FVG haussier : Low[J] > High[J-2] → gap non comblé
    df['fvg_bullish'] = (
        df['Low'] > df['High'].shift(2)
    ).astype(int)

    # FVG baissier : High[J] < Low[J-2] → gap non comblé
    df['fvg_bearish'] = (
        df['High'] < df['Low'].shift(2)
    ).astype(int)

    # === CHANGE OF CHARACTER (CHoCH) ===
    # Rupture du dernier swing high/low = changement de structure
    prev_high = df['High'].rolling(10).max().shift(1)
    prev_low  = df['Low'].rolling(10).min().shift(1)

    df['choch_bullish'] = (
        (df['Close'] > prev_high) &
        (df['Close'].shift(1) <= prev_high.shift(1))
    ).astype(int)

    df['choch_bearish'] = (
        (df['Close'] < prev_low) &
        (df['Close'].shift(1) >= prev_low.shift(1))
    ).astype(int)

    # === LIQUIDITY SWEEP ===
    # Prix dépasse swing puis clôture en sens inverse
    df['liq_sweep_bull'] = (
        (df['Low']  < df['swing_low'].shift(1)) &
        (df['Close'] > df['swing_low'].shift(1))
    ).astype(int)

    df['liq_sweep_bear'] = (
        (df['High'] > df['swing_high'].shift(1)) &
        (df['Close'] < df['swing_high'].shift(1))
    ).astype(int)

    return df.dropna()

df_1d = add_smc_features(df_1d)
df_1h = add_smc_features(df_1h)

# Résumé
smc_cols = ['ssl_distance','bsl_distance','near_ssl','near_bsl',
            'ob_bullish','ob_bearish','fvg_bullish','fvg_bearish',
            'choch_bullish','choch_bearish',
            'liq_sweep_bull','liq_sweep_bear']

print("Features SMC calculées :")
print(f"\nOccurrences sur Daily ({len(df_1d)} jours) :")
for col in ['ob_bullish','ob_bearish','fvg_bullish','fvg_bearish',
            'choch_bullish','choch_bearish',
            'liq_sweep_bull','liq_sweep_bear']:
    n = df_1d[col].sum()
    pct = n/len(df_1d)*100
    print(f"  {col:20s} : {n:4d} fois ({pct:.1f}%)")

Features SMC calculées :

Occurrences sur Daily (1568 jours) :
  ob_bullish           :    2 fois (0.1%)
  ob_bearish           :   16 fois (1.0%)
  fvg_bullish          :  365 fois (23.3%)
  fvg_bearish          :  203 fois (12.9%)
  choch_bullish        :  160 fois (10.2%)
  choch_bearish        :   73 fois (4.7%)
  liq_sweep_bull       :   63 fois (4.0%)
  liq_sweep_bear       :  168 fois (10.7%)


In [8]:
def add_confirmation_features(df):
    """ATR, CVD proxy, divergences RSI et CVD"""
    df = df.copy()

    # === ATR ===
    atr_ind       = ta.volatility.AverageTrueRange(
                        df['High'], df['Low'], df['Close'], window=14)
    df['ATR']     = atr_ind.average_true_range()
    df['ATR_norm'] = df['ATR'] / df['Close']  # ATR normalisé %

    # Régime volatilité
    atr_mean      = df['ATR_norm'].rolling(50).mean()
    df['high_vol'] = (df['ATR_norm'] > 1.5 * atr_mean).astype(int)
    df['low_vol']  = (df['ATR_norm'] < 0.7 * atr_mean).astype(int)

    # Stop dynamique basé ATR
    df['sl_dynamic'] = df['Close'] - 1.5 * df['ATR']
    df['tp_dynamic'] = df['Close'] + 3.0 * df['ATR']

    # === CVD PROXY ===
    # Approximation avec OHLCV
    hl = df['High'] - df['Low']
    hl = hl.replace(0, np.nan)
    df['delta']    = df['Volume'] * (
        (df['Close'] - df['Low']) -
        (df['High']  - df['Close'])
    ) / hl
    df['CVD']      = df['delta'].cumsum()
    df['CVD_norm'] = df['CVD'] / df['CVD'].rolling(50).std()

    # === DIVERGENCES ===
    lookback = 5

    # RSI divergence baissière : prix up, RSI down
    df['div_bear_rsi'] = (
        (df['Close'] > df['Close'].shift(lookback)) &
        (df['RSI_14'] < df['RSI_14'].shift(lookback))
    ).astype(int)

    # RSI divergence haussière : prix down, RSI up
    df['div_bull_rsi'] = (
        (df['Close'] < df['Close'].shift(lookback)) &
        (df['RSI_14'] > df['RSI_14'].shift(lookback))
    ).astype(int)

    # CVD divergence baissière : prix up, CVD down
    df['div_bear_cvd'] = (
        (df['Close'] > df['Close'].shift(lookback)) &
        (df['CVD']   < df['CVD'].shift(lookback))
    ).astype(int)

    # CVD divergence haussière : prix down, CVD up
    df['div_bull_cvd'] = (
        (df['Close'] < df['Close'].shift(lookback)) &
        (df['CVD']   > df['CVD'].shift(lookback))
    ).astype(int)

    return df.dropna()

df_1d = add_confirmation_features(df_1d)
df_1h = add_confirmation_features(df_1h)

print("Features de confirmation calculées :")
print(f"\nATR moyen        : {df_1d['ATR'].mean():.2f} $")
print(f"ATR normalisé    : {df_1d['ATR_norm'].mean()*100:.2f} %")
print(f"Périodes vol haute: {df_1d['high_vol'].sum()} jours")
print(f"\nDivergences RSI bull  : {df_1d['div_bull_rsi'].sum()}")
print(f"Divergences RSI bear  : {df_1d['div_bear_rsi'].sum()}")
print(f"Divergences CVD bull  : {df_1d['div_bull_cvd'].sum()}")
print(f"Divergences CVD bear  : {df_1d['div_bear_cvd'].sum()}")

Features de confirmation calculées :

ATR moyen        : 6.07 $
ATR normalisé    : 1.38 %
Périodes vol haute: 37 jours

Divergences RSI bull  : 28
Divergences RSI bear  : 169
Divergences CVD bull  : 181
Divergences CVD bear  : 146


In [9]:
def add_kill_zones(df):
    """
    Kill Zones basées sur l'heure UTC
    Asian    : 00h-04h
    London   : 07h-10h  ← plus puissante
    NY Open  : 13h-16h  ← deuxième plus puissante
    London Close : 15h-17h
    """
    df = df.copy()

    # Pour données daily — on utilise le jour de la semaine
    if df.index.dtype == 'datetime64[ns]' or hasattr(df.index, 'hour'):
        try:
            hour = df.index.hour
            df['kz_asian']    = ((hour >= 0)  & (hour < 4)).astype(int)
            df['kz_london']   = ((hour >= 7)  & (hour < 10)).astype(int)
            df['kz_ny']       = ((hour >= 13) & (hour < 16)).astype(int)
            df['kz_lc']       = ((hour >= 15) & (hour < 17)).astype(int)
            df['in_kill_zone']= (
                df['kz_london'] | df['kz_ny']
            ).astype(int)

            # Multiplicateur de confiance
            df['kz_multiplier'] = 1.0
            df.loc[df['kz_london']==1, 'kz_multiplier'] = 2.5
            df.loc[df['kz_ny']==1,     'kz_multiplier'] = 2.0
            df.loc[df['kz_lc']==1,     'kz_multiplier'] = 1.5
            df.loc[df['kz_asian']==1,  'kz_multiplier'] = 0.5

        except AttributeError:
            # Données daily — pas d'heure disponible
            df['kz_london']    = 0
            df['kz_ny']        = 0
            df['in_kill_zone'] = 0
            df['kz_multiplier']= 1.0
    return df

df_1d = add_kill_zones(df_1d)
df_1h = add_kill_zones(df_1h)

# Vérification Kill Zones sur 1H
kz_stats = df_1h['in_kill_zone'].value_counts()
london   = df_1h['kz_london'].sum()
ny       = df_1h['kz_ny'].sum()

print("Kill Zones calculées :")
print(f"  London Open (1H) : {london:4d} bougies")
print(f"  NY Open    (1H)  : {ny:4d} bougies")
print(f"  Total Kill Zones : {london+ny:4d} bougies")
print(f"  Hors Kill Zone   : {len(df_1h)-(london+ny):4d} bougies")

Kill Zones calculées :
  London Open (1H) :    0 bougies
  NY Open    (1H)  : 1903 bougies
  Total Kill Zones : 1903 bougies
  Hors Kill Zone   : 3099 bougies


In [10]:
# Vérifier le timezone des données 1H
print("Timezone des données 1H :")
print(f"  Index type : {type(df_1h.index)}")
print(f"  Timezone   : {df_1h.index.tzinfo}")
print(f"\nExemple heures :")
print(df_1h.index[:5].tolist())
print(df_1h.index[100:105].tolist())

# Distribution des heures
hour_dist = df_1h.index.hour.value_counts().sort_index()
print(f"\nDistribution heures UTC :")
print(hour_dist.to_string())

Timezone des données 1H :
  Index type : <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
  Timezone   : UTC

Exemple heures :
[Timestamp('2023-06-14 19:30:00+0000', tz='UTC'), Timestamp('2023-06-15 13:30:00+0000', tz='UTC'), Timestamp('2023-06-15 14:30:00+0000', tz='UTC'), Timestamp('2023-06-15 15:30:00+0000', tz='UTC'), Timestamp('2023-06-15 16:30:00+0000', tz='UTC')]
[Timestamp('2023-07-07 18:30:00+0000', tz='UTC'), Timestamp('2023-07-07 19:30:00+0000', tz='UTC'), Timestamp('2023-07-10 13:30:00+0000', tz='UTC'), Timestamp('2023-07-10 14:30:00+0000', tz='UTC'), Timestamp('2023-07-10 15:30:00+0000', tz='UTC')]

Distribution heures UTC :
Datetime
13    465
14    719
15    719
16    716
17    711
18    711
19    712
20    249


In [11]:
def add_kill_zones_spy(df):
    """
    Kill Zones adaptées à SPY (ETF américain)
    Heures UTC (marché US ouvert 13h30-20h00 UTC)
    
    NY Open KZ     : 13h30-15h00 UTC  ← ouverture US, plus volatile
    NY Midday      : 15h00-17h00 UTC  ← consolidation
    NY Close KZ    : 18h00-20h00 UTC  ← clôture US, forte liquidité
    Power Hour     : 19h00-20h00 UTC  ← dernière heure, très volatile
    """
    df = df.copy()
    hour = df.index.hour
    minute = df.index.minute

    # NY Open Kill Zone — 13h30 à 15h00 UTC
    df['kz_ny_open']  = (
        (hour == 13) & (minute >= 30) |
        (hour == 14)
    ).astype(int)

    # NY Midday — 15h00 à 17h00 UTC
    df['kz_midday']   = (
        (hour >= 15) & (hour < 17)
    ).astype(int)

    # NY Close Kill Zone — 18h00 à 20h00 UTC
    df['kz_ny_close'] = (
        (hour >= 18) & (hour < 20)
    ).astype(int)

    # Power Hour — dernière heure avant clôture
    df['kz_power']    = (hour == 19).astype(int)

    # In Kill Zone = NY Open ou NY Close
    df['in_kill_zone'] = (
        df['kz_ny_open'] | df['kz_ny_close']
    ).astype(int)

    # Multiplicateur de confiance
    df['kz_multiplier'] = 1.0
    df.loc[df['kz_ny_open']==1,  'kz_multiplier'] = 2.5
    df.loc[df['kz_ny_close']==1, 'kz_multiplier'] = 2.0
    df.loc[df['kz_power']==1,    'kz_multiplier'] = 2.2
    df.loc[df['kz_midday']==1,   'kz_multiplier'] = 0.8

    return df

df_1h = add_kill_zones_spy(df_1h)

# Vérification
print("Kill Zones SPY corrigées (UTC) :")
print(f"  NY Open  (13h30-15h00) : {df_1h['kz_ny_open'].sum():4d} bougies")
print(f"  Midday   (15h00-17h00) : {df_1h['kz_midday'].sum():4d} bougies")
print(f"  NY Close (18h00-20h00) : {df_1h['kz_ny_close'].sum():4d} bougies")
print(f"  Power Hr (19h00-20h00) : {df_1h['kz_power'].sum():4d} bougies")
print(f"  Total Kill Zones       : {df_1h['in_kill_zone'].sum():4d} bougies")
print(f"  Multiplicateur max     : {df_1h['kz_multiplier'].max():.1f}x")

# Exemple signaux forts
strong = df_1h[
    (df_1h['in_kill_zone']==1) &
    (df_1h['kz_multiplier']>=2.0)
]
print(f"\nSignaux haute confiance  : {len(strong):4d} bougies")

Kill Zones SPY corrigées (UTC) :
  NY Open  (13h30-15h00) : 1184 bougies
  Midday   (15h00-17h00) : 1435 bougies
  NY Close (18h00-20h00) : 1423 bougies
  Power Hr (19h00-20h00) :  712 bougies
  Total Kill Zones       : 2607 bougies
  Multiplicateur max     : 2.5x

Signaux haute confiance  : 2607 bougies


In [13]:
def compute_zones(df, max_len_strong=20, max_len_weak=5):
    """
    Calcule les zones OB et FVG avec :
    - Longueur dynamique selon la force (volume, taille bougie)
    - Mitigation : zone disparaît quand le prix la touche
    - Retourne une liste de zones à dessiner
    """
    zones = []
    vol_mean = df['Volume'].rolling(20).mean()
    atr      = df['ATR']

    for i in range(2, len(df)):
        row     = df.iloc[i]
        vol_r   = row['Volume'] / vol_mean.iloc[i] if vol_mean.iloc[i] > 0 else 1
        body    = abs(row['Close'] - row['Open'])
        atr_val = atr.iloc[i]

        # Force de la zone — détermine la longueur
        strength = vol_r * (body / atr_val if atr_val > 0 else 1)
        zone_len = int(np.clip(
            max_len_weak + (max_len_strong - max_len_weak) 
            * min(strength / 3, 1),
            max_len_weak, max_len_strong
        ))

        # ── ORDER BLOCK HAUSSIER ──
        if row['ob_bullish'] == 1:
            top    = row['High']
            bottom = row['Open']  # corps de la bougie
            color  = '#26a69a'
            ztype  = 'OB↑'

            # Mitigation — chercher quand prix touche la zone
            end_idx = i + zone_len
            for j in range(i+1, min(i+zone_len+1, len(df))):
                if df.iloc[j]['Low'] <= bottom:
                    end_idx = j
                    break

            zones.append({
                'type'    : ztype,
                'start'   : i,
                'end'     : end_idx,
                'top'     : top,
                'bottom'  : bottom,
                'color'   : color,
                'strength': strength,
                'mitigated': end_idx < i + zone_len
            })

        # ── ORDER BLOCK BAISSIER ──
        if row['ob_bearish'] == 1:
            top    = row['Open']
            bottom = row['Low']
            color  = '#ef5350'
            ztype  = 'OB↓'

            end_idx = i + zone_len
            for j in range(i+1, min(i+zone_len+1, len(df))):
                if df.iloc[j]['High'] >= top:
                    end_idx = j
                    break

            zones.append({
                'type'    : ztype,
                'start'   : i,
                'end'     : end_idx,
                'top'     : top,
                'bottom'  : bottom,
                'color'   : color,
                'strength': strength,
                'mitigated': end_idx < i + zone_len
            })

        # ── FVG HAUSSIER ──
        if row['fvg_bullish'] == 1 and i >= 2:
            top    = row['Low']
            bottom = df.iloc[i-2]['High']
            if top > bottom:
                gap_size = (top - bottom) / atr_val if atr_val > 0 else 1
                zone_len_fvg = int(np.clip(
                    max_len_weak + (max_len_strong - max_len_weak)
                    * min(gap_size, 1),
                    max_len_weak, max_len_strong
                ))
                end_idx = i + zone_len_fvg
                for j in range(i+1, min(i+zone_len_fvg+1, len(df))):
                    if df.iloc[j]['Low'] <= bottom:
                        end_idx = j
                        break
                zones.append({
                    'type'    : 'FVG↑',
                    'start'   : i,
                    'end'     : end_idx,
                    'top'     : top,
                    'bottom'  : bottom,
                    'color'   : '#26a69a',
                    'strength': gap_size,
                    'mitigated': end_idx < i + zone_len_fvg
                })

        # ── FVG BAISSIER ──
        if row['fvg_bearish'] == 1 and i >= 2:
            top    = df.iloc[i-2]['Low']
            bottom = row['High']
            if top > bottom:
                gap_size = (top - bottom) / atr_val if atr_val > 0 else 1
                zone_len_fvg = int(np.clip(
                    max_len_weak + (max_len_strong - max_len_weak)
                    * min(gap_size, 1),
                    max_len_weak, max_len_strong
                ))
                end_idx = i + zone_len_fvg
                for j in range(i+1, min(i+zone_len_fvg+1, len(df))):
                    if df.iloc[j]['High'] >= top:
                        end_idx = j
                        break
                zones.append({
                    'type'    : 'FVG↓',
                    'start'   : i,
                    'end'     : end_idx,
                    'top'     : top,
                    'bottom'  : bottom,
                    'color'   : '#ef5350',
                    'strength': gap_size,
                    'mitigated': end_idx < i + zone_len_fvg
                })

    return zones

# Calculer les zones
zones_1d = compute_zones(df_1d)
zones_1h = compute_zones(df_1h)

print(f"Zones Daily calculées : {len(zones_1d)}")
print(f"  OB↑  : {sum(1 for z in zones_1d if z['type']=='OB↑')}")
print(f"  OB↓  : {sum(1 for z in zones_1d if z['type']=='OB↓')}")
print(f"  FVG↑ : {sum(1 for z in zones_1d if z['type']=='FVG↑')}")
print(f"  FVG↓ : {sum(1 for z in zones_1d if z['type']=='FVG↓')}")
mit = sum(1 for z in zones_1d if z['mitigated'])
print(f"  Mitigées : {mit} ({mit/len(zones_1d)*100:.0f}%)")

Zones Daily calculées : 563
  OB↑  : 2
  OB↓  : 14
  FVG↑ : 356
  FVG↓ : 191
  Mitigées : 354 (63%)


In [ ]:
def plot_smc_chart(df, zones, title, n_bars=90,
                   show_killzones=False, figsize=(20,14)):
    """
    Graphique SMC épuré avec :
    - Bougies japonaises
    - SSL/BSL dynamiques
    - Zones OB/FVG avec longueur dynamique + mitigation
    - RSI Ribbon 3 périodes
    - CVD proxy
    - Kill Zones (si données intraday)
    """
    df_plot = df.tail(n_bars).copy()
    df_plot = df_plot.reset_index(drop=False)
    n       = len(df_plot)

    # Filtrer les zones visibles dans la fenêtre
    offset  = len(df) - n_bars
    visible = [
        z for z in zones
        if z['end'] >= offset and z['start'] <= offset + n_bars
    ]

    # Layout
    if show_killzones:
        height_ratios = [5, 1.5, 1.5, 1.5, 1]
        n_panels      = 5
    else:
        height_ratios = [5, 1.5, 1.5, 1.5]
        n_panels      = 4

    fig = plt.figure(figsize=figsize)
    gs  = gridspec.GridSpec(n_panels, 1,
                            height_ratios=height_ratios,
                            hspace=0.06)
    axes = [fig.add_subplot(gs[i]) for i in range(n_panels)]
    ax1, ax2, ax3, ax4 = axes[0], axes[1], axes[2], axes[3]

    # ── BOUGIES ────────────────────────────────────────────────
    for i, row in df_plot.iterrows():
        c = '#26a69a' if row['Close'] >= row['Open'] else '#ef5350'
        ax1.bar(i, abs(row['Close']-row['Open']),
                bottom=min(row['Open'], row['Close']),
                color=c, width=0.7, alpha=0.9, zorder=2)
        ax1.plot([i,i], [row['Low'], row['High']],
                 color=c, linewidth=0.8, zorder=2)

    # ── SSL / BSL ──────────────────────────────────────────────
    ax1.plot(range(n), df_plot['swing_low'].values,
             color='#ef5350', linewidth=1, linestyle='--',
             alpha=0.6, label='SSL', zorder=3)
    ax1.plot(range(n), df_plot['swing_high'].values,
             color='#26a69a', linewidth=1, linestyle='--',
             alpha=0.6, label='BSL', zorder=3)

    # ── ZONES OB / FVG ─────────────────────────────────────────
    legend_zones = {}
    for z in visible:
        # Coordonnées relatives à la fenêtre
        x_start = max(0,   z['start'] - offset)
        x_end   = min(n-1, z['end']   - offset)
        if x_start >= x_end:
            continue

        alpha_fill = 0.08 if z['mitigated'] else 0.18
        alpha_edge = 0.3  if z['mitigated'] else 0.7
        linestyle  = ':'  if z['mitigated'] else '-'

        # Rectangle de la zone
        rect = plt.Rectangle(
            (x_start, z['bottom']),
            x_end - x_start,
            z['top'] - z['bottom'],
            linewidth  = 0.8,
            edgecolor  = z['color'],
            facecolor  = z['color'],
            alpha      = alpha_fill,
            linestyle  = linestyle,
            zorder     = 1
        )
        ax1.add_patch(rect)

        # Bordure top de la zone
        ax1.plot([x_start, x_end],
                 [z['top'], z['top']],
                 color=z['color'],
                 linewidth=0.8,
                 linestyle=linestyle,
                 alpha=alpha_edge,
                 zorder=3)
        ax1.plot([x_start, x_end],
                 [z['bottom'], z['bottom']],
                 color=z['color'],
                 linewidth=0.8,
                 linestyle=linestyle,
                 alpha=alpha_edge,
                 zorder=3)

        # Label de la zone
        mid_y  = (z['top'] + z['bottom']) / 2
        status = '✓' if z['mitigated'] else ''
        ax1.text(x_end + 0.2, mid_y,
                f"{z['type']}{status}",
                fontsize=6,
                color=z['color'],
                alpha=alpha_edge,
                va='center',
                zorder=4)

        # Légende unique par type
        if z['type'] not in legend_zones:
            legend_zones[z['type']] = mpatches.Patch(
                color=z['color'], alpha=0.4,
                label=z['type'] + (' (actif)' if not z['mitigated']
                                   else ' (mitigé)'))

    # ── CHoCH / BOS ────────────────────────────────────────────
    for i, row in df_plot.iterrows():
        if row.get('choch_bullish', 0) == 1:
            ax1.annotate('CHoCH↑',
                        xy=(i, row['High']),
                        xytext=(i, row['High']*1.004),
                        fontsize=6, color='#26a69a',
                        ha='center', va='bottom',
                        arrowprops=dict(
                            arrowstyle='->', color='#26a69a',
                            lw=1.2),
                        zorder=5)
        if row.get('choch_bearish', 0) == 1:
            ax1.annotate('CHoCH↓',
                        xy=(i, row['Low']),
                        xytext=(i, row['Low']*0.996),
                        fontsize=6, color='#ef5350',
                        ha='center', va='top',
                        arrowprops=dict(
                            arrowstyle='->', color='#ef5350',
                            lw=1.2),
                        zorder=5)

    # ── LIQUIDITY SWEEPS ───────────────────────────────────────
    for i, row in df_plot.iterrows():
        if row.get('liq_sweep_bull', 0) == 1:
            ax1.scatter(i, row['Low']*0.999,
                       marker='*', s=100,
                       color='#26a69a', zorder=6)
        if row.get('liq_sweep_bear', 0) == 1:
            ax1.scatter(i, row['High']*1.001,
                       marker='*', s=100,
                       color='#ef5350', zorder=6)

    # Légende panel 1
    ssl_patch = plt.Line2D([0],[0], color='#ef5350',
                           linewidth=1, linestyle='--',
                           label='SSL')
    bsl_patch = plt.Line2D([0],[0], color='#26a69a',
                           linewidth=1, linestyle='--',
                           label='BSL')
    sweep_b   = plt.Line2D([0],[0], marker='*', color='w',
                           markerfacecolor='#26a69a',
                           markersize=7,
                           label='Liq.Sweep↑')
    sweep_s   = plt.Line2D([0],[0], marker='*', color='w',
                           markerfacecolor='#ef5350',
                           markersize=7,
                           label='Liq.Sweep↓')
    all_legend = [ssl_patch, bsl_patch, sweep_b, sweep_s]
    all_legend += list(legend_zones.values())

    ax1.legend(handles=all_legend, loc='upper left',
               fontsize=6.5, ncol=4,
               facecolor='#161b22',
               edgecolor='#30363d',
               framealpha=0.8)
    ax1.set_title(title, fontsize=11, pad=8)
    ax1.set_ylabel('Prix ($)', fontsize=8)
    ax1.grid(True, alpha=0.2)
    ax1.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v,p: f'${v:.0f}'))

    # ── RSI RIBBON ─────────────────────────────────────────────
    ax2.plot(range(n), df_plot['RSI_7'].values,
             color='#26a69a', linewidth=1,
             label='RSI 7', alpha=0.9)
    ax2.plot(range(n), df_plot['RSI_14'].values,
             color='#EF9F27', linewidth=1,
             label='RSI 14', alpha=0.9)
    ax2.plot(range(n), df_plot['RSI_21'].values,
             color='#ef5350', linewidth=1,
             label='RSI 21', alpha=0.9)

    ax2.fill_between(range(n),
                     df_plot['RSI_7'].values,
                     df_plot['RSI_21'].values,
                     where=df_plot['RSI_7'].values > df_plot['RSI_21'].values,
                     alpha=0.12, color='#26a69a')
    ax2.fill_between(range(n),
                     df_plot['RSI_7'].values,
                     df_plot['RSI_21'].values,
                     where=df_plot['RSI_7'].values < df_plot['RSI_21'].values,
                     alpha=0.12, color='#ef5350')

    # Score RSI en fond
    for i, score in enumerate(df_plot['RSI_score'].values):
        if score == 3:
            ax2.axvspan(i-0.5, i+0.5,
                       alpha=0.12, color='#26a69a')
        elif score == -3:
            ax2.axvspan(i-0.5, i+0.5,
                       alpha=0.12, color='#ef5350')

    for level, color in [(70,'#ef5350'),(50,'#8b949e'),(30,'#26a69a')]:
        ax2.axhline(level, color=color, linewidth=0.6,
                   linestyle=':', alpha=0.5)

    ax2.set_ylim(0, 100)
    ax2.set_ylabel('RSI', fontsize=8)
    ax2.legend(loc='upper left', fontsize=6.5, ncol=3,
               facecolor='#161b22', edgecolor='#30363d',
               framealpha=0.8)
    ax2.grid(True, alpha=0.2)

    # ── CVD ────────────────────────────────────────────────────
    cvd = df_plot['CVD_norm'].values
    ax3.plot(range(n), cvd,
             color='#b87af5', linewidth=1,
             label='CVD Proxy')
    ax3.fill_between(range(n), cvd, 0,
                     where=cvd > 0,
                     alpha=0.15, color='#26a69a')
    ax3.fill_between(range(n), cvd, 0,
                     where=cvd < 0,
                     alpha=0.15, color='#ef5350')

    # Divergences
    for i, row in df_plot.iterrows():
        if row.get('div_bull_cvd',0)==1:
            ax3.scatter(i, cvd[i], marker='^',
                       s=50, color='#26a69a', zorder=5)
        if row.get('div_bear_cvd',0)==1:
            ax3.scatter(i, cvd[i], marker='v',
                       s=50, color='#ef5350', zorder=5)

    ax3.axhline(0, color='#8b949e',
                linewidth=0.6, alpha=0.5)
    ax3.set_ylabel('CVD', fontsize=8)
    ax3.legend(loc='upper left', fontsize=6.5,
               facecolor='#161b22', edgecolor='#30363d',
               framealpha=0.8)
    ax3.grid(True, alpha=0.2)

    # ── ATR ────────────────────────────────────────────────────
    atr = df_plot['ATR_norm'].values * 100
    ax4.fill_between(range(n), atr, 0,
                     color='#EF9F27', alpha=0.35)
    ax4.plot(range(n), atr,
             color='#EF9F27', linewidth=0.8)
    ax4.axhline(np.mean(atr),
                color='#EF9F27', linewidth=0.8,
                linestyle='--', alpha=0.6)

    for i, row in df_plot.iterrows():
        if row.get('high_vol',0)==1:
            ax4.axvspan(i-0.5, i+0.5,
                       alpha=0.12, color='#ef5350')

    ax4.set_ylabel('ATR%', fontsize=8)
    ax4.grid(True, alpha=0.2)

    # ── KILL ZONES panel 5 ─────────────────────────────────────
    if show_killzones and n_panels == 5:
        ax5 = axes[4]
        kz_mul = df_plot['kz_multiplier'].values \
                 if 'kz_multiplier' in df_plot.columns \
                 else np.ones(n)

        colors_kz = []
        for v in kz_mul:
            if   v >= 2.5: colors_kz.append('#26a69a')
            elif v >= 2.0: colors_kz.append('#EF9F27')
            elif v >= 1.5: colors_kz.append('#b87af5')
            elif v <= 0.8: colors_kz.append('#30363d')
            else:          colors_kz.append('#8b949e')

        ax5.bar(range(n), kz_mul,
                color=colors_kz, alpha=0.7, width=0.8)
        ax5.axhline(1, color='#8b949e',
                   linewidth=0.6, linestyle='--')
        ax5.set_ylabel('KZ ×', fontsize=8)
        ax5.set_ylim(0, 3)
        ax5.grid(True, alpha=0.2)

        kz_legend = [
            mpatches.Patch(color='#26a69a', label='NY Open ×2.5'),
            mpatches.Patch(color='#EF9F27', label='NY Close ×2.0'),
            mpatches.Patch(color='#b87af5', label='Power Hr ×2.2'),
            mpatches.Patch(color='#30363d', label='Midday ×0.8'),
        ]
        ax5.legend(handles=kz_legend, loc='upper right',
                  fontsize=6, ncol=4,
                  facecolor='#161b22',
                  edgecolor='#30363d')

        # Colorier panel prix selon Kill Zone
        for i, (mul, row) in enumerate(
                zip(kz_mul, df_plot.itertuples())):
            if mul >= 2.0:
                ax1.axvspan(i-0.5, i+0.5,
                           alpha=0.05,
                           color='#26a69a', zorder=0)

    # ── DATES ──────────────────────────────────────────────────
    last_ax   = axes[-1]
    step      = max(1, n // 12)
    tick_pos  = list(range(0, n, step))
    date_col  = 'Date' if 'Date' in df_plot.columns \
                else df_plot.columns[0]
    tick_lbl  = []
    for i in tick_pos:
        try:
            d = df_plot[date_col].iloc[i]
            tick_lbl.append(pd.Timestamp(d).strftime('%d/%m'))
        except:
            tick_lbl.append(str(i))

    last_ax.set_xticks(tick_pos)
    last_ax.set_xticklabels(tick_lbl,
                             rotation=45, fontsize=7)
    for ax in axes[:-1]:
        plt.setp(ax.get_xticklabels(), visible=False)

    plt.savefig(f"../models/{title.replace(' ','_')}.png",
                bbox_inches='tight',
                facecolor='#0d1117', dpi=150)
    plt.show()
    print(f"Graphique sauvegardé !")

print("Fonction graphique prête !")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2883090470.py, line 199)